# 📁 PPO 매매정책 강화학습 + Advanced RAG 챗봇 — 통합 실습

## 이 실습의 전체 구조

```
                    ┌────────────────────────────┐
                    │   주(主) — PPO 강화학습      │
                    │   "매수/매도/유보"를 결정한다  │
                    └────────────────────────────┘
                                  │
                                  │ 결정을 내린 뒤
                                  ▼
                    ┌─────────────────────────────┐
                    │   부(副) — Advanced RAG     │
                    │   "왜 그런 결정인지" 과거 사례 │
                    │   + 뉴스/공시로 거들기 위해    │
                    │   호출된다                   │
                    └─────────────────────────────┘
                                  │
                                  ▼
                    최종 답변: "매수 / 매도 / 유보 + 근거 설명"
```

**PPO가 두뇌이고, RAG는 그 두뇌의 결정을 설명하는 보조 자료 검색입니다.**
RAG가 먼저 검색해서 PPO에게 알려주는 구조가 아니라, **PPO가 먼저 결론(매수/매도/유보)을
내고, 그 결론을 RAG가 부연 설명**하는 순서입니다. 이 순서를 반대로 착각하지 않도록
본 실습 전체에서 이 위계를 유지합니다.

---

## 📂 전체 폴더 구성 (실습이 끝나면 이렇게 됩니다)

지금 폴더에는 **이 노트북과 `data/`만 있습니다.** 나머지 파일/폴더는
전부 이번 실습에서 Claude Code로 직접 만듭니다.

```
01_golden_cross/
├── (이 노트북)                       ← 지금 보고 있는 이 파일 (이미 있음)
├── data/                            ← 이미 있음
│   └── {종목코드}.csv               (종목코드, 일자, 시가, 종가, 고가, 저가)
│
├── indicators/                      ★ STEP 2에서 직접 만들기
│   └── technical.py                 (SMA/EMA/골든·데드크로스/이격도/RSI)
│
├── env/                             ★ STEP 3에서 직접 만들기
│   └── trading_env.py               (TradingEnv, gymnasium.Env)
│
├── train/                           ★ STEP 4·5에서 직접 만들기
│   ├── reward.py                    (reward 셰이핑)
│   └── run_training.py              (PPO 학습 → models/ 저장)
│
├── models/                          (자동 생성)
│   └── policy_ppo.zip               (여러 종목으로 학습된 단일 정책망)
│
├── inference/                       ★ STEP 6에서 직접 만들기
│   └── predict.py                   (오늘 state → action + 확률)
│
├── explain/                         ★ STEP 6·7·8에서 직접 만들기
│   ├── rule_based.py                (수치 근거 산출, LLM 미사용)
│   ├── rag_retriever.py             (Advanced RAG 검색 — 4기법)
│   └── llm_explainer.py             (수치+RAG 근거 → 자연어 설명)
│
├── corpus/                          ★ STEP 7에서 직접 만들기
│   ├── build_corpus.py              (뉴스/공시 수집 + 사례 카드 + 임베딩)
│   ├── news_cache/                  (자동 생성, 뉴스/공시 캐시)
│   └── chroma_db/                   (자동 생성, 벡터DB)
│
└── app.py                           ★ STEP 9에서 직접 만들기 (Streamlit 챗봇)
```

---

## 🎯 배경 및 목적

종목별로 분리된 일자별 시세(종목코드, 일자, 시가, 종가, 고가, 저가) 데이터를 사용해서,
**"지금 이 지표 상태 + 내 포지션에서 매수/매도/유보 중 무엇을 해야 자산가치가
늘어나는가"를 학습하는 PPO 강화학습 에이전트**를 만듭니다. 이것이 이번 실습의
**본체(主)** 입니다.

> ⚠️ 이건 "2일 후 가격이 오를지 내릴지 맞히는 퀴즈"가 아닙니다. 포지션·자산가치
> 개념이 있는 **매매 정책**을 학습합니다. 차이는 STEP 1에서 다시 설명합니다.

이 PPO 에이전트가 결정을 내린 뒤, **Advanced RAG(ChromaDB + hybrid search +
reranking)를 부수적으로 호출**해서 "왜 그런 판단을 했는지"를 과거 유사 사례와
당시 뉴스/공시까지 곁들여 설명합니다. RAG는 결정을 내리지 않으며, PPO가 이미
내린 결정에 곁들이는 자연어 설명 자료를 찾아오는 역할만 합니다.

```
data/{종목코드}.csv
        │
        ▼
indicators (지표 계산) ──▶ PPO 강화학습 (★ 주) ──▶ 매수 / 매도 / 유보 결정
                                  │
                                  ▼ (결정이 내려진 뒤에만 호출됨)
                          Advanced RAG 검색 (부) ──▶ 과거 유사 사례 + 당시 뉴스/공시
                                  │
                                  ▼
                  최종 답변 = PPO의 결정 + RAG의 설명 (챗봇 형태로 출력)
```

---

## 📊 PPO 설계 핵심 요약 (본체)

| 항목 | 값 |
|---|---|
| State | SMA5/20/60 이격률, EMA20, 골든·데드크로스 플래그, 이격도20(중심화), RSI14(정규화), 포지션, 미실현손익 — 9차원, lookback 20일 시퀀스 |
| Action | 0=매도, 1=유보, 2=매수 |
| Reward | 자산가치 변화율 − 거래비용 ± 지표 기반 셰이핑(RSI 과매수 매수 페널티, 골든크로스 보유 보너스, 이격도 과열 매수 페널티) |
| 학습 단위 | **모델 1개** — 여러 종목의 episode를 섞어서 단일 정책망 학습 (종목별 모델 아님) |
| 지표·episode 경계 | 종목별로 **반드시 분리** (학습 큐에 넣을 때만 섞음) |
| 알고리즘/라이브러리 | PPO (stable-baselines3), 정책망 MLP `[64,64]`로 시작 |

### 데이터 — 시세 외에 추가로 필요한 것

| 항목 | 필요 여부 |
|---------------------------------|-------------------------------|
| 종목코드/일자/시가/종가/고가/저가 | **필수** (이미 `data/`에 있음) |
| 거래량 | 선택 — 있으면 신호 품질 향상, 없어도 진행 가능 |
| 초기 자본금/거래 단위 | 데이터가 아니라 **시뮬레이션 설정값** |
| 거래비용/수수료율 | 데이터가 아니라 **가정값** (기본 0.2~0.3%) |


---
## 🧠 잠깐 — 이 모델은 "방향 예측"이 아니라 "매매 정책"입니다

```
방향 예측형 (이 실습에서 쓰지 않는 방식)        매매 정책형 (이 실습의 방식)
──────────────────────────────              ──────────────────────────
"2일 후 가격이 오를까 내릴까?"                  "지금 포지션과 지표 상태에서
 맞히면 +10, 틀리면 -10                          어떤 행동을 해야 내 계좌
                                                (자산가치)가 불어나는가?"

포지션 개념 없음 — 매 스텝이                    실제 현금↔주식 전환,
독립적인 "퀴즈"                                 거래비용, 미실현손익이
                                                전부 시뮬레이션됨

보상 = 예측 정답 여부                           보상 = 자산가치 변화율
       (분류 문제에 가까움)                            (진짜 트레이딩 시뮬레이션)
```

방향 예측형은 "맞고 틀리고"만 보기 때문에 실제로 사고팔았을 때 이득인지는
보장하지 못합니다. 이 실습은 **자산가치 자체를 보상으로 쓰기 때문에, 학습이
끝나면 "오늘 매수해야 하는가"에 곧바로 답할 수 있습니다.**

또한 RAG가 검색할 corpus를 만들 때도 같은 차이가 적용됩니다 — 이 실습에서는
"오늘 PPO가 무엇을 판단했는가 + 그 판단의 수치 근거"를 사례 카드로 만들지,
"2일 후 방향이 맞았는가"를 사례 카드로 만들지 않습니다.


---
## 🖥 실습 순서

### STEP 0 · 폴더 이동 + 파일 확인

이 노트북이 있는 폴더(`ppo_rag_trading/` 또는 `01_golden_cross/`)에서
아래 셀을 실행해서 현재 상태를 확인합니다.

In [ ]:
!ls
# 이 노트북 파일과 data/ 폴더만 있으면 정상

In [ ]:
!ls data/
# 005930.csv  000660.csv  ...  종목코드.csv 형태로 있는지 확인

**새 터미널 탭을 열고:**

```bash
claude
```

이후 STEP들의 프롬프트는 이 노트북에 markdown 코드블록으로 정리해두었습니다.
**그대로 복사해서 Claude Code 터미널에 붙여넣고** 진행하세요. 노트북 자체에서
PPO 학습이나 Streamlit 실행을 직접 돌리기보다는, Claude Code가 만든 모듈을
이 노트북에서 `import`해서 결과를 확인하는 방식으로 사용하는 걸 권장합니다.

---
### STEP 1 · [Claude Code 실습 — 설계 합의] 프로젝트 구조 결정

아직 코드는 만들지 않습니다. Claude Code와 폴더/모듈 설계를 먼저 합의합니다.

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
나는 종목별 일자별 주식 시세 데이터를 갖고 있어. data/{종목코드}.csv 형태이고,
각 파일은 [일자, 시가, 종가, 고가, 저가] 컬럼을 가져 (날짜순 정렬).

이 데이터로 PPO(stable-baselines3) 강화학습 에이전트를 만들어서
"매수 / 매도 / 유보" 3가지 행동을 결정하게 하고 싶어.

핵심 원칙:
- 이 모델은 "2일 후 가격 방향을 맞히는 퀴즈"가 아니라, "지금 포지션과 지표
  상태에서 어떤 행동을 해야 자산가치가 늘어나는가"를 학습하는 매매 정책이야.
- 모델은 하나만 학습하고, 여러 종목의 데이터를 합쳐서 학습 데이터로 사용해.
  단, 지표 계산과 episode(포지션·손익 시뮬레이션) 구성은 반드시 종목별로
  분리해서 계산하고, 종목 경계를 넘어 섞이지 않게 해줘. 학습 단계에서만
  여러 종목의 episode를 섞어서 모델에 입력해.
- state에는 종목코드 자체를 포함하지 않아 — 모델이 지표 패턴의 일반 규칙을
  배우게 하려는 목적이야.
- 이 에이전트는 추후 RAG 시스템과 결합될 예정이니, 예측 결과와 판단 근거를
  추적 가능한 형태로 남겨줘.

state(9차원, lookback 20일 시퀀스):
SMA5/20/60 이격률, EMA20, 골든크로스 플래그, 데드크로스 플래그,
이격도20(100 중심화), RSI14(정규화), 현재 포지션(보유여부), 미실현손익

reward: 자산가치 변화율 - 거래비용(기본 0.2~0.3%, config화)
  + RSI>70인데 매수 시 페널티(-0.1)
  + 골든크로스 직후 보유 유지 시 보너스(+0.05)
  + 이격도>110인데 신규 매수 시 페널티(-0.1)
  (가중치는 전부 config로 분리, on/off 플래그도 추가)

action: Discrete(3) = 0매도/1유보/2매수

알고리즘: PPO, 정책망은 SB3 기본 MlpPolicy, hidden layer [64,64]로 시작.
추후 LSTM 정책망으로 교체 비교할 수 있도록 policy_kwargs를 분리된 config로 빼줘.

다음 모듈로 나눠서 작업하자 (폴더는 프로젝트 루트 기준):
- indicators/technical.py : 기술지표 계산 (종목별 분리 계산)
- env/trading_env.py : gymnasium TradingEnv (종목별 episode 경계 분리)
- train/reward.py : reward 셰이핑 함수
- train/run_training.py : PPO 학습 — 여러 종목 episode를 섞어서 단일 모델
  학습, models/policy_ppo.zip 저장
- inference/predict.py : 학습된 모델로 임의 종목의 "오늘" 행동 + 확률 예측
- explain/rule_based.py : 수치 근거 산출 (LLM 미사용, 결정론적)
- (RAG와 챗봇은 이후 STEP에서 별도로 설계할 예정이니 지금은 폴더만 잡아두자)

코드 작성 전에 먼저:
1. 폴더/파일 구조 (위 목록 기준으로 더 다듬을 부분 있으면 제안해줘)
2. 각 모듈의 입출력 인터페이스 (함수 시그니처 수준)
3. train/run_training.py에서 "여러 종목 episode를 섞는" 구체적 방식
4. 모델(.zip)이 아직 없을 때와 있을 때 이후 만들 앱이 어떻게 다르게 동작해야
   하는지 — 학습은 별도 CLI 스크립트로 분리하고 앱은 추론만 담당할 것
5. 데이터 분할 전략 (시간 기준 train/validation, 종목 내에서 룩어헤드 누설 방지)

을 markdown으로 정리해서 보여줘. 내가 확인하면 순서대로 구현 시작하자.
```

> 💡 이 단계에서 Claude Code가 제안하는 "여러 종목 episode를 섞는 방식"을
> 잘 읽어보세요. 종목별 모델을 따로 두는 게 아니라 **모델 1개**로 일반화된
> 정책을 학습하는 게 이 실습의 핵심 설계입니다.


---
### STEP 2 · [Claude Code 실습 — 주(主)] 기술지표 계산 모듈

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
indicators/technical.py를 작성해줘.

요구사항:
1. 입력: 종목코드/일자/시가/종가/고가/저가 컬럼을 가진 pandas DataFrame
   (data/{종목코드}.csv 들을 종목코드 컬럼을 붙여 합친 형태라고 가정)
2. 다음 컬럼을 추가해서 반환:
   - SMA5, SMA20, SMA60
   - EMA20 (span=20)
   - golden_flag, dead_flag: 직전 N개 캔들 내 SMA5가 SMA20을 상향/하향
     돌파했는지 (0/1)
   - disparity20: (종가/SMA20)*100, 중심화 버전(disparity20_centered)도 같이
   - RSI14: Wilder's smoothing 방식, 분모 0(하락 없는 구간) 예외처리 포함
   - RSI14_norm: (RSI14-50)/50
3. 종목코드별로 그룹화해서 계산하고, 종목 경계를 넘어 지표가 섞이지 않게
   할 것 (단위테스트로 반드시 검증: 두 종목을 이어붙인 가상 데이터를 만들어서
   경계 지점의 지표가 다른 종목 값에 영향받지 않는지 확인)
4. 초기 구간(이동평균 계산 안 되는 첫 N일) NaN 처리는 옵션(drop/backfill)으로
5. docstring + pytest 단위테스트 (RSI는 알려진 예시값으로 검증)

테스트 통과 확인 후 결과 보여줘.
```

In [ ]:
!pytest indicators/ -v

---
### STEP 3 · [Claude Code 실습 — 주(主)] Gym 환경 만들기

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
env/trading_env.py를 작성해줘.

state 벡터 (9차원, lookback 20일 시퀀스 → shape (20, 9)):
[SMA5_disp, SMA20_disp, SMA60_disp, golden_flag, dead_flag,
 disparity20_centered, RSI14_norm, position, unrealized_pnl]

- position: 보유=1, 미보유=0
- unrealized_pnl: 보유 중일 때 (현재 종가-매수가)/매수가, 미보유면 0

action: Discrete(3) — 0=매도, 1=유보, 2=매수
- 미보유인데 매도 선택, 보유 중인데 매수 선택 등 무효 행동의 처리 방식을
  제안해줘 (무시 / 작은 페널티 등)

거래 제약:
- 매수는 현금 있을 때만, 매도는 보유 주식 있을 때만 유효
- 거래비용 비율 파라미터화 (기본 0.2~0.3%)

핵심 — episode 경계:
- 환경은 반드시 "단일 종목" 데이터로만 구성되어야 함 (여러 종목을 한
  episode 안에서 이어붙이지 말 것)
- reset() 시 어떤 종목, 어떤 시작 시점으로 episode를 구성할지는 외부
  (train 모듈)에서 주입받는 구조로 설계 (종목 리스트를 섞는 로직은 env가
  아니라 train에서 담당)

episode 종료: 데이터 끝 도달 또는 자산이 일정 비율 이하로 하락 시 종료

reset(), step() 구현, render()는 텍스트로 현재 상태 출력.
random policy로 여러 episode 굴려보는 스모크 테스트 작성, 특히 종목이
바뀔 때 포지션/손익이 올바르게 초기화되는지 검증.

파일명: env/trading_env.py
```

In [ ]:
!python -m env.trading_env
# 또는 작성된 스모크 테스트 스크립트 실행

---
### STEP 4 · [Claude Code 실습 — 주(主)] Reward 셰이핑

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
train/reward.py를 작성해줘.

기본 보상: R_t = (V_t - V_{t-1})/V_{t-1} - 거래비용
 (V_t = 현금 + 보유주식 평가액)

셰이핑 추가:
R_t_final = R_t
  - 0.1 * [RSI>70인데 매수한 경우]
  + 0.05 * [골든크로스 직후 매수 포지션 유지하는 경우]
  - 0.1 * [이격도>110인데 신규 매수한 경우]

요구사항:
- 가중치(0.1, 0.05, 0.1, 거래비용율)는 전부 config로 분리, 튜닝 가능하게
- 셰이핑 on/off 플래그 추가 (순수 수익 기반 reward와 비교 실험용)
- 각 셰이핑 항이 episode 동안 몇 번 발동했는지 카운트 로깅
  (에이전트가 지표 규칙만 흉내내고 있는지 점검용)
- env/trading_env.py의 step()에서 이 reward 함수를 사용하도록 연결해줘

파일명: train/reward.py
```

---
### STEP 5 · [Claude Code 실습 — 주(主)] PPO 학습 실행

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
train/run_training.py를 작성해줘.

핵심 — 여러 종목 episode를 섞어서 단일 모델 학습:
- data/ 안의 모든 종목 CSV를 읽어서, 종목별로 시간 기준 train/validation
  분리 (예: 특정 날짜 이전 train, 이후 validation)
- train 구간 데이터로 종목별 episode를 독립적으로 생성한 뒤, 학습 시점에는
  종목 순서가 섞이도록 구성 (env reset마다 다른 종목이 무작위로 선택되거나,
  DummyVecEnv/SubprocVecEnv로 여러 종목 환경을 동시에 굴리는 방식 등 —
  STEP 1에서 합의한 방식대로 구현)
- 어떤 방식을 택했는지 코드 주석과 실행 로그에 남겨줘

정책망: MlpPolicy, policy_kwargs={"net_arch": [64, 64]}를 config로 분리
(추후 LSTM 정책망으로 쉽게 교체할 수 있게 구조화)

하이퍼파라미터(learning_rate, n_steps, batch_size, gamma 등)는
config/argparse로.

학습 중 validation set 성과(누적수익률, MDD, 샤프비율, 승률)를 주기적으로
평가, tensorboard 로깅. 가장 좋은 validation 성과의 체크포인트를
models/policy_ppo.zip 으로 저장.

학습 완료 후 train/validation 수익률 곡선을 차트로 저장.
종목별 학습 결과(종목코드, 평균 episode return, 검증 적중 경향)를
요약하는 표를 마지막에 출력.

먼저 종목 2~3개로 빠르게 50000 스텝 smoke run을 돌려서 파이프라인이
에러 없이 동작하는지 확인하고, 그 다음 본 학습 설정을 논의하자.

파일명: train/run_training.py
```

In [ ]:
!python train/run_training.py
# [smoke run] 종목 2~3개, 50000 step...
# episode 10  avg_return=...
# ...
# 학습 완료 → models/policy_ppo.zip 저장
#
# ===== 검증 구간 요약 =====
# 누적수익률   MDD   샤프비율   매수/유보/매도 비율
# ...

> 예상 소요 시간: smoke run CPU 기준 약 3~10분, 본 학습은 종목 수와
> 스텝 수에 비례해 수십 분~수시간

이 결과 요약표는 "전체적으로 잘 학습됐는지"를 숫자로만 보여줍니다. 하지만
**내가 만든 모델이 실제로 한 종목에 대해 어떤 판단을 내리는지**는 아직
본 적이 없습니다. STEP 6에서 직접 확인해봅니다.

---
### STEP 6 · [Claude Code 실습 — 주(主)] 추론 + 수치 근거 — 직접 눈으로 확인하기

지금까지는 숫자(episode return, 샤프비율)로만 결과를 봤습니다. 여기서는
**방금 학습시킨 PPO 모델 하나를 직접 불러와서, 진짜로 매수/매도/유보 중
무엇을 답하는지, 그리고 왜 그런지(수치 근거)를 화면에 띄워봅니다.**
RAG는 아직 등장하지 않습니다 — 순수하게 PPO + 수치 근거만 확인하는 단계입니다.

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
inference/predict.py 와 explain/rule_based.py를 작성해줘.

inference/predict.py:
1. 입력: 종목코드 (data/{종목코드}.csv 기준)
2. 해당 종목 최근 데이터(lookback 20일 이상) 로드
3. indicators/technical.py로 지표 계산 → state 시퀀스 구성
4. models/policy_ppo.zip 로드 → action 예측
   (모델 파일이 없으면 "먼저 train/run_training.py를 실행하세요" 안내 후 종료)
5. 반환: action, action_probs(매수/유보/매도 확률), 사용된 raw 지표값 전부,
   기준일자

explain/rule_based.py — LLM 미사용, 100% 결정론적:
predict.py 출력을 받아서:
- RSI14가 과매수(≥70)/과매도(≤30)/중립 중 어디인지, 정확한 값
- disparity20이 과열(≥105)/침체(≤95)/중립 중 어디인지, 정확한 값
- golden_flag/dead_flag 최근 발생 여부
- reward shaping 발동 조건 매칭 여부 (RSI>70 매수 패턴 등 플래그)
- action_probs 1위·2위 차이가 10%p 이내면 "확신도 낮음" 플래그

반환값은 dataclass/dict로. 모든 값은 데이터에서 직접 계산, 추측 없음.
RSI=75, disparity=112 가상 케이스로 플래그가 올바르게 켜지는지 단위테스트.

마지막으로, 위 두 모듈을 연결해서 검증 구간(학습에 쓰지 않은 뒤쪽 20%)
중 임의 날짜 5곳을 뽑아 "그날 종가 흐름 표 + PPO 판단 + 수치 근거"를
화면에 출력하는 간단한 점검 스크립트 check_prediction.py도 같이 만들어줘.

출력 형식 예시:
  ──────────────────────────────────────
  [점검 1/5] 005930 (2025-06-01 기준)
  최근 5일 종가 흐름: 47,300 → 47,690 → 48,099 → 48,283 → 48,232원
  PPO 판단: 매수 (확률 62% / 유보 31% / 매도 7%)
  수치 근거: RSI14=58.2(중립), 이격도20=102.1(중립),
            골든크로스 4일전 발생, 확신도: 보통
  ──────────────────────────────────────

파일명: inference/predict.py, explain/rule_based.py, check_prediction.py
```

In [ ]:
!pytest explain/ -v

In [ ]:
# check_prediction.py 실행 결과를 노트북에서 바로 확인하고 싶다면
# (Claude Code가 위 파일들을 다 만든 후에 실행하세요)
!python check_prediction.py --ticker 005930


예상 출력 예시:

```
──────────────────────────────────────
[점검 1/5] 005930 (2025-06-01 기준)
최근 5일 종가 흐름: 47,300 → 47,690 → 48,099 → 48,283 → 48,232원
PPO 판단: 매수 (확률 62% / 유보 31% / 매도 7%)
수치 근거: RSI14=58.2(중립), 이격도20=102.1(중립),
          골든크로스 4일전 발생, 확신도: 보통
──────────────────────────────────────
```

> 💡 이 단계의 의미: STEP 5의 요약표는 "전체 평균"이라 감이 잘 안 옵니다.
> STEP 6처럼 **특정 날짜, 특정 가격, 특정 판단을 직접 눈으로 보면** "내가
> 만든 모델이 진짜로 작동하는구나"를 체감할 수 있습니다. 이후 STEP 7~9에서
> 만드는 RAG/챗봇도 결국 이 PPO 판단 위에 설명을 덧붙이는 것입니다.

**여기까지가 본체(主) 완성입니다.** 이미 이 PPO 모델만으로도 "매수/매도/유보"를
결정하고 수치 근거까지 직접 눈으로 확인할 수 있습니다. 이어지는 STEP 7~9는
이 결정에 "설명"을 덧붙이는 부수 단계입니다.

---
### STEP 6-1 · (부수 단계) 왜 RAG가 필요한가, 그리고 왜 "Advanced"인가

PPO는 "매수"라는 결정과 수치 근거(RSI=75 등)는 내놓지만, **"과거에 비슷한
상황이 있었는지, 그때 어떻게 됐는지"는 알지 못합니다.** 그날의 뉴스/공시가
어떤 맥락이었는지도 PPO 내부에는 없습니다. 이 역할을 RAG가 맡습니다.

```
PPO + rule_based가 못 하는 것              Advanced RAG가 하는 것
──────────────────────────────────────────────────────────
"과거에 이런 패턴이 있었나?"                 과거 사례 카드 + 당시 뉴스/공시를
"그때는 어떻게 됐나?"                        검색해서 근거로 제시
```

일반 RAG(벡터 검색 한 번)로는 부족한 이유:

| 문제 | 일반 RAG의 한계 | Advanced RAG의 해법 |
|---|---|---|
| "RSI 70"처럼 정확한 수치 조건 검색이 약함 | 벡터 검색만으로는 의미는 비슷해도 숫자 조건을 놓침 | **Hybrid Search** (키워드+벡터 결합) |
| 질문 하나로는 검색 커버리지가 좁음 | "지금 사도 될까?"라는 한 문장만으론 관련 사례를 다 못 찾음 | **Query Expansion** (여러 변형 질의로 확장) |
| 다른 종목·자기 자신의 데이터가 섞여 들어옴 | 의미적으로만 비슷한 게 아니라 조건도 맞아야 함 | **Metadata Filtering** (종목/수치구간/날짜 필터) |
| 1차 검색 결과에 노이즈가 많음 | 벡터 유사도 1위가 실제로는 관련성이 떨어질 수 있음 | **Reranking** (정밀 재정렬 모델로 최종 선별) |

단, **RAG가 PPO보다 먼저 호출되거나, RAG의 검색 결과가 PPO의 결정을 바꾸는
일은 없습니다.** 순서는 항상 "PPO 결정 → 수치 근거 → RAG 검색 → 결합"입니다.


---
### STEP 7 · [Claude Code 실습 — 부(副)] RAG Corpus 구축 + Advanced RAG 검색

이 단계는 두 부분으로 나뉩니다. **7-A(corpus 구축)는 1회성 배치 작업**이고,
**7-B(검색 모듈)는 매 질문마다 호출되는 부분**입니다.

#### STEP 7-A · Corpus 구축 (뉴스/공시 수집 + 사례 카드 + 임베딩)

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
corpus/build_corpus.py를 작성해줘.

1. 뉴스/공시 수집 (외부 데이터, 신규 수집 필요):
   - 종목코드 + 날짜를 입력하면 그 날짜 전후(예: -1일~+1일) 뉴스 기사
     제목/요약을 가져오는 함수. 사용 가능한 뉴스 API가 있는지 먼저 물어보고,
     없으면 네이버 금융 뉴스 페이지 등에서 수집하는 방법을 제안해줘
     (크롤링 시 약관/rate limit 준수 여부도 같이 짚어줘)
   - 같은 종목코드 + 날짜로 DART(전자공시시스템) Open API를 통해 공시
     제목/보고서명을 가져오는 함수 (무료 키 발급이 필요하다는 점을 안내하고,
     키가 없을 때는 이 부분을 빈 값으로 두고 진행 가능하게 처리)
   - 수집한 뉴스/공시는 종목코드+날짜를 키로 캐싱
     (corpus/news_cache/{종목코드}_{일자}.json)

2. 지표 사례 카드 생성:
   - indicators/technical.py 와 inference/predict.py 결과를 이용해서,
     종목·일자별로 다음을 하나의 "사례 카드" 텍스트로 합성:
     "{종목코드} {일자}: RSI14={값}({구간}), 이격도20={값}({구간}),
      골든크로스 발생={yes/no}, 데드크로스 발생={yes/no}, 당시 PPO 판단={action}
      (확률 {p}%), 관련 뉴스: {뉴스제목 요약}, 관련 공시: {공시제목}.
      이후 5일 수익률={%}, 10일 수익률={%}"
   - 사후 수익률(5일/10일 후)은 그 시점 이후 실제 시세에서 계산 — 이건
     "학습"에 쓰는 게 아니라 "사후 기록"으로 사례 카드에 남기는 것이므로
     데이터 누설이 아니라는 점을 주석으로 명시해줘
   - 뉴스/공시가 없는 날짜는 "관련 뉴스 없음"으로 명시 (LLM이 없는 걸
     지어내지 않도록)

3. 임베딩 + 벡터DB 저장:
   - 각 사례 카드 텍스트를 임베딩 모델로 벡터화 (로컬 모델 vs API 기반
     옵션을 제안하고 trade-off 설명)
   - corpus/chroma_db에 ChromaDB로 저장: document(사례카드 텍스트),
     embedding(벡터), metadata(종목코드, 일자, RSI14, disparity20,
     golden_flag, dead_flag, PPO_action, PPO_확률, 5일후수익률, 10일후수익률)
   - 키워드 검색(BM25)용 인덱스도 별도 구축 (rank_bm25 라이브러리 등 제안)

4. 증분 업데이트: 이미 corpus에 있는 종목·일자는 재계산하지 않고 스킵

먼저 종목 1~2개, 최근 1개월 정도의 작은 범위로 corpus를 만들어서 구조가
잘 동작하는지 확인하고, 그 다음 data/ 전체 종목으로 확장하자.

파일명: corpus/build_corpus.py
```

In [ ]:
!python corpus/build_corpus.py
# [corpus 구축] 005930, 2025-06-01 ~ 2025-06-30 (1개월 테스트 범위)
# 뉴스 수집: ... / 공시 수집: ...
# 사례 카드 생성: 20건
# ChromaDB 저장 완료 → corpus/chroma_db/
# BM25 인덱스 구축 완료

> 💡 뉴스/공시 API에 키가 없으시면 일단 "뉴스/공시 없음" 상태로 진행해도
> 됩니다 — 지표 사례 카드(RSI/이격도/크로스/PPO판단/사후수익률)만으로도
> 다음 단계의 RAG는 정상 동작합니다. 외부 데이터는 준비되는 대로 증분
> 업데이트로 채우면 됩니다.

#### STEP 7-B · Advanced RAG 검색 모듈

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
explain/rag_retriever.py를 작성해줘. STEP 7-A에서 만든 corpus(ChromaDB +
BM25 인덱스)를 검색하는 Advanced RAG 모듈. 다음 4가지 기법을 전부 포함:

1. Query Expansion:
   - 입력: explain/rule_based.py에서 나온 수치 근거
   - 활성화된 조건들(RSI 과매수, 골든크로스 등)을 각각 별도 검색 질의로
     확장 (예: "RSI 과매수 매수 사례", "골든크로스 직후 보유 사례" 등)
   - LLM으로 확장할지, 규칙 기반 템플릿으로 확장할지 선택지를 제시하고
     비용/속도 trade-off를 설명해줘 (매 추론마다 호출되니 속도가 중요함을
     감안해서 추천해줘)

2. Hybrid Search:
   - 확장된 질의들 각각에 대해 벡터 검색(ChromaDB)과 키워드 검색(BM25)을
     동시에 수행, 결과를 점수 정규화 후 결합 (weighted sum 또는 RRF 중
     더 적합한 방법을 골라서 이유를 설명해줘)

3. Metadata Filtering:
   - 같은 종목 우선 vs 다른 종목의 비슷한 패턴도 포함 여부를 옵션으로
   - RSI/disparity20 값이 현재 상황과 ±N 범위 내인 사례만 남기는 필터
   - 너무 최근(lookback 기간과 겹치는) 사례는 제외 — 자기참조 방지
   - 필터 후 후보가 너무 적으면(3건 미만) 단계적으로 완화하는 fallback

4. Reranking:
   - 필터링된 후보를 현재 상황(수치 근거 텍스트)과의 관련도로 재정렬
   - cross-encoder 기반 재정렬 모델을 제안하고, 가벼운 대안(코사인 유사도
     재계산)과의 trade-off도 설명
   - 최종 Top-K(기본 5건)만 다음 단계로 전달

반환값: 검색된 사례 카드 텍스트 리스트 + metadata + 최종 관련도 점수.

성능을 위해 임베딩 모델과 reranking 모델은 모듈 로드 시 한 번만 초기화
(매 호출마다 재로딩 금지)하도록 캐싱 구조로 작성해줘.

단위테스트: mock corpus(가상 사례 카드 10건)로 4가지 기법이 각각 의도대로
동작하는지 — metadata filter가 조건에 안 맞는 사례를 실제로 걸러내는지,
reranking 전후로 순위가 바뀌는지 검증.

파일명: explain/rag_retriever.py
```

In [ ]:
!pytest explain/rag_retriever* -v

---
### STEP 8 · [Claude Code 실습 — 주+부 결합] LLM 자연어 설명

이제 PPO(주)의 결정 + 수치 근거 + RAG(부)의 검색 결과를 하나의 자연어
설명으로 묶습니다.

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
explain/llm_explainer.py를 작성해줘. Claude API로 explain/rule_based.py
(수치 근거) + explain/rag_retriever.py(RAG 검색 결과)를 받아 자연어로 변환.

이 순서는 반드시 지켜줘 — PPO가 먼저 결정을 내리고 수치 근거가 확정된
뒤에만 RAG를 호출하며, RAG 검색 결과가 PPO의 결정 자체를 바꾸지 않는다는
게 핵심이야. 이 함수는 이미 정해진 action을 "설명"만 할 뿐, 다시 판단하지 않아.

시스템 프롬프트에 반드시 포함:
"너는 주어진 수치 데이터와 검색된 과거 사례만 보고 설명을 작성한다.
데이터에 없는 값을 추측하거나 새로운 판단을 만들어내지 않는다. 검색된
과거 사례를 인용할 때는 반드시 그 사례의 종목코드와 날짜를 함께 언급해서
출처를 명확히 한다. 검색된 사례가 없으면 '참고할 과거 사례를 찾지
못했습니다'라고 명시하고 지어내지 않는다. 확신도가 낮게 표시된 경우 그
사실을 분명히 언급한다."

입력:
- rule_based 결과: action, action_probs, RSI14/disparity20 값과 구간,
  golden/dead flag, shaping 플래그, 확신도 낮음 플래그
- rag_retriever 결과: 재정렬된 Top-K 과거 사례 카드(종목코드, 날짜, 당시
  지표값, 당시 PPO판단, 사후 수익률, 관련 뉴스/공시 텍스트)

출력: 4~6문장 한국어 설명. 현재 판단 근거를 먼저 설명하고, 그 다음
"과거 비슷한 사례에서는 ~"로 RAG 근거를 붙이는 구조. 인용한 사례의
종목코드+날짜를 괄호로 표기 (예: "(005930, 2025-06-05)")

추가 요구사항:
- API 실패 시 수치 근거 + 사례 목록만으로 구성한 기본 템플릿 설명으로
  자동 폴백 (부수 기능 실패가 본체 기능에 영향을 주지 않아야 함)
- LLM 응답에 등장하는 모든 수치가 실제 값과 일치하는지 검증, 불일치 시
  폴백 템플릿 사용
- LLM 응답에 등장하는 종목코드+날짜 인용이 실제로 검색결과에 존재하는
  사례인지 검증 (없는 사례를 지어내 인용하는 것 방지)
- mock 데이터로 LLM 호출 없이 폴백 템플릿이 동작하는지 테스트

파일명: explain/llm_explainer.py
```

In [ ]:
!pytest explain/llm_explainer* -v

In [ ]:
# 전체 파이프라인을 노트북에서 한 번에 확인하고 싶다면
# (Claude Code가 모든 모듈을 만든 후에 실행하세요)
from inference.predict import predict
from explain.rule_based import analyze
from explain.rag_retriever import retrieve
from explain.llm_explainer import explain

r = predict('005930')
rb = analyze(r)
rag = retrieve(rb)
print(explain(rb, rag))


예상 출력 예시:

```
[종목 005930] 예측 결과: 매수 (확률 62%)
  - 기준일: 2025-06-05 / 종가: 47,690원
  - RSI14=58.2(중립), 이격도20=102.1(중립), 골든크로스 4일전 발생
  - 참고 근거 (Advanced RAG 검색 결과, 2건):
      1. (005930, 2024-11-12) 비슷한 골든크로스 패턴, 이후 5일 +2.1%
      2. (000660, 2025-02-20) RSI 중립대 매수 후 10일 +3.4%
  - 종합 설명: 현재 RSI와 이격도는 중립 구간이며 4일 전 골든크로스가
    발생해 추세 추종 보너스 조건에 부합합니다. 과거 유사 사례
    (005930, 2024-11-12) 등에서도 비슷한 패턴 이후 단기 상승으로
    이어진 경향이 있었으나, 이는 참고 사례일 뿐 결과를 보장하지 않습니다.
```

---
### STEP 9 · [Claude Code 실습] Streamlit 챗봇으로 완성하기

마지막으로, 지금까지 만든 모든 모듈을 대화형 챗봇으로 묶습니다.

아래 프롬프트를 **그대로 복사해서 Claude Code에 붙여넣기** 하세요:

```text
app.py를 Streamlit 챗봇 형태로 작성해줘. st.chat_message / st.chat_input
기반으로, 사용자가 자연어로 묻고 답을 받는 대화형 UI.

앞에서 만든 inference/predict.py, explain/rule_based.py,
explain/rag_retriever.py, explain/llm_explainer.py를 그대로 사용해줘.
이 순서는 반드시 지켜줘 — PPO가 먼저 결정을 내리고, RAG는 그 결정을
설명하는 보조 역할만 한다는 게 핵심이야. RAG 검색 결과가 PPO의 결정을
바꾸지 않도록 해줘.

핵심 동작 흐름 (사용자가 메시지를 보낼 때마다):
1. 사용자 입력에서 종목코드 또는 종목명을 추출. 종목을 특정할 수 없으면
   "어느 종목에 대해 궁금하신가요?"라고 되묻기 (직전 메시지에서 종목이
   언급됐으면 이어서 사용)
2. inference/predict.py → action, action_probs, 지표값
3. explain/rule_based.py → 수치 근거
4. explain/rag_retriever.py → Advanced RAG 검색
5. explain/llm_explainer.py → 최종 자연어 답변
6. 답변을 채팅 형태로 출력. 답변 아래에 인용된 과거 사례를 expander로
   펼쳐서 원문(사례 카드 텍스트, 뉴스/공시 출처) 확인 가능하게

UI 구성:
- 사이드바: 종목코드/종목명 직접 입력 selectbox, 모델 로딩 상태 표시,
  "대화 초기화" 버튼
- 메인 채팅창: 행동(매수/매도/유보)을 색상 배지로 강조, action_probs는
  작은 막대로 함께 표시
- "참고한 과거 사례" expander: Top-K 사례를 종목코드/날짜/요약/사후수익률과
  함께 나열
- 모델(.zip) 부재 시: 채팅 입력을 막고 "학습된 모델이 없습니다.
  train/run_training.py를 먼저 실행해주세요" 안내만 표시
- 대화 시작 시 첫 안내 메시지: "종목코드나 종목명을 입력하시면 현재
  매수/매도/유보 판단과 과거 유사 사례를 근거로 설명해드립니다.
  (예: '삼성전자 지금 사도 될까?', '005930 어때?')"
- 하단 고정 안내: "이 답변은 과거 데이터 기반 모델의 참고 지표이며 투자
  자문이 아닙니다. 실제 매매는 본인 판단과 책임 하에 결정하세요."

대화 맥락 처리:
- st.session_state로 메시지 히스토리 유지
- 같은 종목에 대한 후속 질문("왜 그렇게 판단했어?") 은 재추론 없이 직전
  결과를 컨텍스트로 LLM에 넘겨서 답변, 새 종목 언급 시에는 처음부터
  다시 추론 — 이 구분을 어떻게 판별할지 방법을 제안해줘

상태 관리: 모델/데이터/RAG 인덱스 로딩은 st.cache_resource
에러 처리: 존재하지 않는 종목코드, lookback보다 짧은 데이터, RAG 검색
결과 0건(이 경우 "참고할 과거 사례를 찾지 못했습니다"로 명시)

실행 방식: streamlit run app.py
파일명: app.py
```

In [ ]:
!streamlit run app.py

브라우저가 열리면 채팅창에 다음처럼 입력해서 확인해보세요:

```
사용자: 삼성전자 지금 사도 될까?

어시스턴트: [매수 🔴]  매수 62% / 유보 31% / 매도 7%

현재 RSI14는 58.2로 중립 구간이며, 이격도20도 102.1로 과열되지 않은
상태입니다. 4일 전 골든크로스가 발생해 추세 추종 보너스 조건에
부합합니다. 과거 유사 사례(005930, 2024-11-12)에서도 비슷한 패턴
이후 단기 상승 경향이 있었으나, 이는 참고 사례일 뿐 결과를
보장하지 않습니다.

▸ 참고한 과거 사례 (펼치기)
  1. 005930 · 2024-11-12 · 5일후 +2.1% · 10일후 +3.8%
  2. 000660 · 2025-02-20 · 5일후 +3.4% · 10일후 +5.1%

이 답변은 과거 데이터 기반 모델의 참고 지표이며 투자 자문이 아닙니다.
```

---
## 🔗 전체 데이터 흐름 한눈에 보기

```
[주(主) 트랙 — PPO]                         [부(副) 트랙 — Advanced RAG]

data/{종목코드}.csv                          data/{종목코드}.csv +
      │                                      뉴스/공시 API
      ▼                                            │
indicators/technical.py                            ▼
      │                                      corpus/build_corpus.py
      ▼                                      (사례 카드 + 임베딩)
env/trading_env.py                                  │
      │                                            ▼
train/run_training.py                       corpus/chroma_db/
      │                                      (벡터DB + BM25 인덱스)
      ▼                                            │
models/policy_ppo.zip                              │
      │                                            │
      ▼                                            │
inference/predict.py                               │
      │                                            │
      ▼                                            │
explain/rule_based.py (★ 수치 근거 직접 확인)         │
      │                                            │
      └─────────────────┬──────────────────────────┘
                         ▼
              explain/rag_retriever.py
              (query expansion → hybrid search →
               metadata filtering → reranking)
                         │
                         ▼
              explain/llm_explainer.py
                         │
                         ▼
                      app.py (챗봇)
                         │
                         ▼
        "[종목] 판단: 매수/매도/유보 + 근거 설명 + 참고 사례"
```


---
## ❓ 자주 묻는 질문

**Q. 이 모델이 "2일 후 방향을 맞히는" 모델과 뭐가 다른가요?**
```
방향 예측형은 포지션 개념 없이 "맞았는지/틀렸는지"만 보상으로 씁니다.
이 실습의 모델은 실제 현금↔주식 전환, 거래비용, 미실현손익까지
시뮬레이션해서 "자산가치가 실제로 늘었는지"를 보상으로 씁니다.
그래서 학습이 끝나면 "오늘 매수해야 하는가"에 곧바로 답할 수 있습니다.
(자세한 비교는 위 "🧠 잠깐" 섹션 참고)
```

**Q. RAG 없이 PPO만으로도 동작하나요?**
```
네. inference/predict.py + explain/rule_based.py만으로도 "매수/매도/유보
+ 수치 근거"는 완성됩니다 (STEP 6에서 확인). RAG(STEP 7~8)는 여기에
"과거 유사 사례"라는 추가 설명을 덧붙이는 부수 기능입니다.
check_prediction.py는 RAG를 전혀 사용하지 않습니다.
```

**Q. 뉴스/공시 API 키가 없는데 진행할 수 있나요?**
```
네. corpus/build_corpus.py는 뉴스/공시가 없으면 "관련 뉴스 없음"으로
명시하고 지표 사례 카드(RSI/이격도/크로스/PPO판단/사후수익률)만으로
corpus를 만듭니다. Advanced RAG의 4가지 기법 모두 이 상태로도
정상 동작합니다. 키가 준비되면 build_corpus.py를 다시 실행해서
증분 업데이트하면 됩니다.
```

**Q. 종목별로 모델을 따로 학습하는 게 아닌가요?**
```
아닙니다. 이 실습은 모델 1개를 여러 종목의 episode를 섞어서 학습합니다.
단, 지표 계산(STEP 2)과 episode 구성(STEP 3)은 종목별로 반드시 분리해서
계산하고, 학습 큐에 넣을 때만(STEP 5) 섞습니다. 종목코드 자체는
state에 포함하지 않아서, 학습에 없던 신규 종목에도 같은 모델을 쓸 수
있습니다.
```

**Q. action_probs의 확률을 얼마나 믿어도 되나요?**
```
PPO policy가 내놓는 확률이 실제 신뢰도와 정확히 일치한다는 보장은
없습니다(calibration 문제). 1위/2위 확률 차이가 10%p 이내면
explain/rule_based.py가 "확신도 낮음"으로 플래그를 켜도록 설계했으니,
이 플래그가 켜진 경우는 신호로 과신하지 마세요.
```

**Q. RAG로 검색된 과거 사례가 있으면 이번에도 같은 결과가 나오나요?**
```
아닙니다. "참고 사례"일 뿐 "예측의 증거"가 아닙니다. 과거에 비슷한
패턴이 있었다는 것이 이번에도 같은 결과를 보장하지 않습니다.
explain/llm_explainer.py의 시스템 프롬프트도 이 전제를 항상 포함하도록
설계했습니다.
```

**Q. GPU가 없는데 가능한가요?**
```
PPO 학습과 임베딩 계산 모두 CPU로 가능합니다 (단, 시간이 더 걸립니다).
GPU 환경에서 실행하면 훨씬 빠릅니다.
```

---

## ⚠️ 주의사항 — 투자 자문이 아닙니다

이 시스템은 과거 데이터 기반 PPO 모델의 판단과 RAG로 검색된 과거
유사사례를 보여줄 뿐, 어느 쪽도 미래 수익을 보장하지 않습니다.
실거래 적용 전 충분한 out-of-sample 검증과 paper trading 기간을
거치시길 권장합니다.

---

> 💡 이 통합 실습의 핵심 포인트
> **PPO가 주(主), Advanced RAG는 부(副)** — 이 순서를 기억하세요.
> PPO가 먼저 "매수/매도/유보"를 결정하고, RAG는 그 결정에 과거 사례와
> 뉴스/공시 근거를 덧붙일 뿐입니다. RAG가 먼저 검색하거나 PPO의 결정을
> 바꾸는 구조가 아니라는 점이 이 시스템 설계에서 가장 중요하게 기억해야
> 할 위계입니다.
> 그리고 STEP 6처럼, **학습은 숫자로 끝나는 게 아니라 직접 눈으로
> 확인하는 것**까지가 한 세트라는 것도 함께 기억하세요.
